In [0]:
%sql
WITH manifest AS (
    SELECT *
    FROM workspace.migration_dev.gold_load_manifest
    WHERE batch_id = 'batch_001'
),
acknowledgements AS (
    SELECT *
    FROM workspace.migration_dev.load_acknowledgements
    WHERE batch_id = 'batch_001'
),
reconciliation AS (
    SELECT
        COALESCE(m.document_id, a.document_id) AS document_id,
        CASE
            WHEN m.document_id IS NULL
                THEN 'UNEXPECTED_ACKNOWLEDGEMENT'
            WHEN a.document_id IS NULL
                THEN 'MISSING_ACKNOWLEDGEMENT'
            WHEN NOT (m.validation_run_id <=> a.validation_run_id)
                THEN 'OUTDATED_ACKNOWLEDGEMENT'
            WHEN a.load_status IS NULL OR a.load_status <> 'SUCCESS'
                THEN 'LOAD_NOT_SUCCESSFUL'
            WHEN a.destination_path IS NULL OR a.loaded_at IS NULL
                THEN 'INCOMPLETE_ACKNOWLEDGEMENT'
            WHEN NOT (m.source_sha256 <=> a.destination_sha256)
                THEN 'DESTINATION_CHECKSUM_MISMATCH'
            ELSE 'RECONCILED'
        END AS reconciliation_status
    FROM manifest m
    FULL OUTER JOIN acknowledgements a
        ON m.batch_id = a.batch_id
       AND m.document_id = a.document_id
)
SELECT
    reconciliation_status,
    COUNT(*) AS document_count
FROM reconciliation
GROUP BY reconciliation_status
ORDER BY reconciliation_status;

reconciliation_status,document_count
RECONCILED,100


In [0]:
import uuid
from pyspark.sql import functions as F

batch_id = "batch_001"

controls = (
    spark.table("workspace.migration_dev.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(controls) != 1:
    raise RuntimeError("Expected one batch control record")

control = controls[0]
approval_run = control["validation_run_id"]
expected = control["expected_document_count"]

manifest = (
    spark.table("workspace.migration_dev.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
)

acks = (
    spark.table("workspace.migration_dev.load_acknowledgements")
    .filter(F.col("batch_id") == batch_id)
)

manifest_count = manifest.count()
ack_count = acks.count()

# Duplicate identities must prevent completion.
unique_keys = True
for data in [manifest, acks]:
    if data.groupBy("document_id").count().filter(
        F.col("count") > 1
    ).limit(1).count() > 0:
        unique_keys = False

comparison = (
    manifest.alias("m")
    .join(
        acks.alias("a"),
        ["batch_id", "document_id"],
        "full_outer",
    )
    .select(
        F.col("document_id"),
        (
            F.col("m.validation_run_id").eqNullSafe(approval_run)
            & F.col("a.validation_run_id").eqNullSafe(approval_run)
            & F.col("a.load_status").eqNullSafe("SUCCESS")
            & F.col("a.destination_path").isNotNull()
            & F.col("a.loaded_at").isNotNull()
            & F.col("m.source_sha256").isNotNull()
            & F.col("m.source_sha256").eqNullSafe(
                F.col("a.destination_sha256")
            )
        ).alias("is_reconciled"),
    )
)

summary = comparison.agg(
    F.count("*").alias("total"),
    F.count_if(F.col("is_reconciled")).alias("matched"),
).first()

matched = summary["matched"]
unresolved = summary["total"] - matched

complete = (
    control["batch_status"] == "APPROVED"
    and control["failed_checks"] == 0
    and expected > 0
    and unique_keys
    and manifest_count == expected
    and ack_count == expected
    and matched == expected
    and unresolved == 0
)

completion_status = "COMPLETE" if complete else "INCOMPLETE"
reconciliation_run_id = str(uuid.uuid4())

result = spark.createDataFrame(
    [(
        reconciliation_run_id,
        batch_id,
        approval_run,
        expected,
        manifest_count,
        ack_count,
        matched,
        unresolved,
        completion_status,
    )],
    """
    reconciliation_run_id STRING,
    batch_id STRING,
    validation_run_id STRING,
    expected_document_count BIGINT,
    manifest_document_count BIGINT,
    acknowledgement_count BIGINT,
    reconciled_document_count BIGINT,
    unresolved_document_count BIGINT,
    load_completion_status STRING
    """,
).withColumn("reconciled_at", F.current_timestamp())

(
    result.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.migration_dev.downstream_reconciliation")
)

display(
    spark.table("workspace.migration_dev.downstream_reconciliation")
    .filter(F.col("reconciliation_run_id") == reconciliation_run_id)
)

if not complete:
    raise RuntimeError("Downstream reconciliation is incomplete")

print(f"{batch_id}: COMPLETE")

reconciliation_run_id,batch_id,validation_run_id,expected_document_count,manifest_document_count,acknowledgement_count,reconciled_document_count,unresolved_document_count,load_completion_status,reconciled_at
d87e1a20-bf1f-43c8-8341-1f3ee2a6f82a,batch_001,4e66d857-a136-4b69-af88-a9acada6bd83,100,100,100,100,0,COMPLETE,2026-10-14T10:48:15.475Z


batch_001: COMPLETE
